# Convert the SWOT 10-second KML orbit to a 1-second CSV

This notebook reads the SWOT nominal-orbit KML and creates a headerless CSV compatible with `new_swot_science_orbit.csv`:

```text
track,longitude,latitude
```

KML pieces split at the antimeridian are merged by track. Artificial `±179.999999°` seam vertices are removed before spherical interpolation from 10-second to 1-second spacing.

In [ ]:
from collections import OrderedDict
from pathlib import Path
from xml.etree import ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Select the input and output files

The default output name avoids overwriting the existing orbit. Change it to `new_swot_science_orbit.csv` after checking the result if you want to replace that file.

In [ ]:
repository_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "satpass").is_dir()
)
orbit_directory = repository_root / "src" / "satpass" / "data" / "nominal_tracks"
kml_path = orbit_directory / "swot_science_orbit_feb2025-v1.0_10s_nadir.kml"
csv_path = orbit_directory / "new_swot_science_orbit_from_kml_1s.csv"

print(f"Input:  {kml_path}")
print(f"Output: {csv_path}")

## Read and merge KML track pieces

The same track can occur in multiple placemarks when a line crosses the antimeridian. Document order is retained when those pieces are joined.

In [ ]:
namespace = {"kml": "http://www.opengis.net/kml/2.2"}
root = ET.parse(kml_path).getroot()
track_pieces = OrderedDict()

for placemark in root.findall(".//kml:Placemark", namespace):
    track = int(placemark.findtext("kml:name", namespaces=namespace))
    coordinate_text = placemark.findtext(".//kml:coordinates", namespaces=namespace)
    if not coordinate_text:
        continue

    coordinates = np.array(
        [[float(value) for value in token.split(",")[:2]] for token in coordinate_text.split()],
        dtype=float,
    )
    track_pieces.setdefault(track, []).append(coordinates)

tracks_10s = OrderedDict()
removed_seam_points = 0
for track, pieces in track_pieces.items():
    coordinates = np.vstack(pieces)
    # GDAL inserted these boundary vertices when splitting lines at ±180°.
    seam = np.isclose(np.abs(coordinates[:, 0]), 179.999999, atol=2e-6)
    removed_seam_points += int(seam.sum())
    coordinates = coordinates[~seam]

    # Remove an exact repeated vertex if two KML pieces share one.
    if len(coordinates) > 1:
        repeated = np.r_[False, np.all(np.isclose(np.diff(coordinates, axis=0), 0), axis=1)]
        coordinates = coordinates[~repeated]
    tracks_10s[track] = coordinates

print(f"Tracks: {len(tracks_10s)}")
print(f"10-second points: {sum(map(len, tracks_10s.values())):,}")
print(f"Removed antimeridian seam points: {removed_seam_points:,}")

## Interpolate from 10 seconds to 1 second

Interpolation is performed on three-dimensional unit vectors using spherical linear interpolation. This avoids longitude discontinuities at the antimeridian and behaves well near the poles. Each original 10-second interval becomes ten 1-second intervals.

In [ ]:
def lonlat_to_unit_vector(coordinates):
    lon = np.deg2rad(coordinates[:, 0])
    lat = np.deg2rad(coordinates[:, 1])
    return np.column_stack((np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)))


def unit_vector_to_lonlat(vectors):
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    longitude = np.rad2deg(np.arctan2(vectors[:, 1], vectors[:, 0]))
    latitude = np.rad2deg(np.arctan2(vectors[:, 2], np.hypot(vectors[:, 0], vectors[:, 1])))
    return np.column_stack((longitude, latitude))


def interpolate_track_1s(coordinates, source_step_seconds=10):
    if len(coordinates) < 2:
        return coordinates.copy()

    vectors = lonlat_to_unit_vector(coordinates)
    start = vectors[:-1]
    end = vectors[1:]
    omega = np.arccos(np.clip(np.einsum("ij,ij->i", start, end), -1.0, 1.0))
    fractions = np.arange(source_step_seconds, dtype=float) / source_step_seconds

    sin_omega = np.sin(omega)
    denominator = np.where(np.abs(sin_omega) > 1e-12, sin_omega, 1.0)
    weight_start = np.sin((1.0 - fractions[None, :]) * omega[:, None]) / denominator[:, None]
    weight_end = np.sin(fractions[None, :] * omega[:, None]) / denominator[:, None]
    interpolated = weight_start[..., None] * start[:, None, :] + weight_end[..., None] * end[:, None, :]

    nearly_equal = np.abs(sin_omega) <= 1e-12
    if nearly_equal.any():
        interpolated[nearly_equal] = (
            (1.0 - fractions)[None, :, None] * start[nearly_equal, None, :]
            + fractions[None, :, None] * end[nearly_equal, None, :]
        )

    vectors_1s = np.vstack((interpolated.reshape(-1, 3), vectors[-1]))
    return unit_vector_to_lonlat(vectors_1s)


tracks_1s = OrderedDict(
    (track, interpolate_track_1s(coordinates))
    for track, coordinates in tracks_10s.items()
)
print(f"1-second points: {sum(map(len, tracks_1s.values())):,}")

## Build and validate the SatPass table

In [ ]:
orbit_1s = pd.concat(
    [
        pd.DataFrame(
            {"track": track, "longitude": coordinates[:, 0], "latitude": coordinates[:, 1]}
        )
        for track, coordinates in tracks_1s.items()
    ],
    ignore_index=True,
)

assert orbit_1s["track"].nunique() == 584
assert orbit_1s["longitude"].between(-180, 180).all()
assert orbit_1s["latitude"].between(-90, 90).all()
assert np.isfinite(orbit_1s[["longitude", "latitude"]]).all().all()

print(orbit_1s.shape)
orbit_1s.head()

## Compare the original and interpolated points

In [ ]:
example_track = 2
original = tracks_10s[example_track]
interpolated = tracks_1s[example_track]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(interpolated[:, 0], interpolated[:, 1], linewidth=0.8, label="1-second interpolation")
ax.scatter(original[:, 0], original[:, 1], s=8, color="tab:red", label="KML 10-second points")
ax.set(xlabel="Longitude", ylabel="Latitude", title=f"SWOT track {example_track}")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()

## Write the headerless CSV

In [ ]:
orbit_1s.to_csv(csv_path, index=False, header=False, float_format="%.8f")
print(f"Wrote {len(orbit_1s):,} rows to {csv_path}")
print(f"File size: {csv_path.stat().st_size / 1024**2:.1f} MiB")